# 08 — Structured LightGBM Hyperparameter Search

## Objective

With the 04.5 feature set fixed, this notebook performs a small, hypothesis-driven parameter search on the established 70/30 holdout. Trials are staged to separate tree complexity from sampling and regularization effects.

Each completed trial is appended immediately to CSV, making the search resumable after interruption. Full three-fold CV is reserved for a short list of candidates to control compute cost and repeated validation exposure.


In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    """Use the parent of the notebook working directory as the project root."""
    project_root = Path.cwd().resolve().parent
    if not (project_root / 'data').is_dir():
        raise FileNotFoundError(
            "Expected the project data directory one level above the current working directory."
        )
    return project_root

import gc, json, platform, time
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
print('Python:',platform.python_version()); print('LightGBM:',lgb.__version__)


## 1. Fix the data and validation split

All trials use the same customer features, indices, categorical handling, metric implementation, and random seed. Fixing the evaluation surface makes trial differences interpretable and prevents split noise from being mistaken for parameter improvement.


In [ ]:
ROOT=find_project_root()
STEP_DIR=ROOT/'08_hyperparameter_tuning'
DATA_PATH=ROOT/'04_5_temporal_refined'/'train_aggregated_temporal_refined.feather'
HOLDOUT_METRICS_PATH=ROOT/'04_5_temporal_refined'/'temporal_refined_metrics.json'
RESULTS_PATH=STEP_DIR/'lightgbm_tuning_results.csv'
BEST_MODEL_PATH=STEP_DIR/'lightgbm_tuning_best.txt'
BEST_PREDICTIONS_PATH=STEP_DIR/'lightgbm_tuning_best_validation_predictions.parquet'
SUMMARY_PATH=STEP_DIR/'lightgbm_tuning_summary.json'
assert DATA_PATH.exists() and HOLDOUT_METRICS_PATH.exists()
data=pd.read_feather(DATA_PATH)
assert data['customer_ID'].is_unique and data.shape[1]-2==1022
customer_ids=data['customer_ID'].copy(); y=data['target'].astype('int8').copy()
X=data.drop(columns=['customer_ID','target']).copy(); del data; gc.collect()
X['S_2_last']=(pd.to_datetime(X['S_2_last'])-pd.Timestamp('2018-03-01')).dt.days.astype('int16')
CAT_VARS=['B_30','B_38','D_114','D_116','D_117','D_120','D_126','D_63','D_64','D_66','D_68']
CAT_FEATURES=[f'{c}_last' for c in CAT_VARS]
for c in CAT_FEATURES: X[c]=X[c].astype('category')
indices=np.arange(len(X))
train_idx,valid_idx=train_test_split(indices,train_size=.70,test_size=.30,random_state=42,stratify=y)
y_train=y.iloc[train_idx]; y_valid=y.iloc[valid_idx]; valid_ids=customer_ids.iloc[valid_idx].to_numpy()
REFERENCE=json.loads(HOLDOUT_METRICS_PATH.read_text())['temporal_refined']
print(f'Train {len(train_idx):,}; valid {len(valid_idx):,}; features {X.shape[1]}'); print('04.5 reference:',REFERENCE)


## 2. Define a resumable trial runner

The runner merges trial-specific changes with base parameters, trains with early stopping, evaluates ROC-AUC and the AMEX metric, records runtime, and writes the result immediately.

Existing completed trial IDs are skipped. This checkpoint pattern is useful for long laptop experiments because partial progress remains valid.


In [ ]:
def amex_metric(y_true,y_pred):
    y_true=np.asarray(y_true,dtype=np.int8); y_pred=np.asarray(y_pred,dtype=np.float64); w=np.where(y_true==0,20.,1.)
    order=np.argsort(-y_pred,kind='mergesort'); yy,ww=y_true[order],w[order]; top=np.cumsum(ww)<=.04*ww.sum()
    top_four=(yy[top]*ww[top]).sum()/(y_true*w).sum()
    def gini(pred):
        idx=np.argsort(-pred,kind='mergesort'); sy,sw=y_true[idx],w[idx]
        return np.sum((np.cumsum(sy*sw)/np.sum(sy*sw)-np.cumsum(sw)/np.sum(sw))*sw)
    return .5*(gini(y_pred)/gini(y_true)+top_four)

BASE_PARAMS=dict(objective='binary',n_estimators=2500,learning_rate=.03,max_depth=-1,num_leaves=31,min_child_samples=100,subsample=1.0,subsample_freq=0,colsample_bytree=.8,reg_alpha=0.,reg_lambda=1.,max_bin=127,random_state=42,n_jobs=4,verbosity=-1)
PARAM_COLUMNS=['num_leaves','min_child_samples','subsample','subsample_freq','colsample_bytree','reg_alpha','reg_lambda','max_bin','n_estimators','learning_rate']
RESULT_COLUMNS=['trial_id','stage','status',*PARAM_COLUMNS,'roc_auc','amex_metric','best_iteration','training_seconds','delta_auc_vs_04_5','delta_amex_vs_04_5','hit_iteration_cap']

def load_results():
    return pd.read_csv(RESULTS_PATH) if RESULTS_PATH.exists() else pd.DataFrame(columns=RESULT_COLUMNS)

def run_trial(trial_id,stage,changes):
    existing=load_results()
    if trial_id in set(existing.get('trial_id',[])):
        print('SKIP completed:',trial_id); return existing.loc[existing.trial_id==trial_id].iloc[-1].to_dict()
    params=BASE_PARAMS|changes
    print('\nSTART',trial_id,stage,{k:params[k] for k in PARAM_COLUMNS})
    started=time.perf_counter(); model=lgb.LGBMClassifier(**params)
    model.fit(X.iloc[train_idx],y_train,eval_set=[(X.iloc[valid_idx],y_valid)],eval_metric='auc',categorical_feature=CAT_FEATURES,callbacks=[lgb.early_stopping(100,verbose=True),lgb.log_evaluation(100)])
    probability=model.predict_proba(X.iloc[valid_idx],num_iteration=model.best_iteration_)[:,1]
    elapsed=time.perf_counter()-started; auc=float(roc_auc_score(y_valid,probability)); amex=float(amex_metric(y_valid,probability))
    row={'trial_id':trial_id,'stage':stage,'status':'completed',**{k:params[k] for k in PARAM_COLUMNS},'roc_auc':auc,'amex_metric':amex,'best_iteration':int(model.best_iteration_),'training_seconds':float(elapsed),'delta_auc_vs_04_5':auc-REFERENCE['roc_auc'],'delta_amex_vs_04_5':amex-REFERENCE['amex_metric'],'hit_iteration_cap':bool(model.best_iteration_>=params['n_estimators']-100)}
    updated=pd.concat([existing,pd.DataFrame([row])],ignore_index=True); updated.to_csv(RESULTS_PATH,index=False)
    eligible=updated[updated.roc_auc>=REFERENCE['roc_auc']-.0001]
    best_id=(eligible if len(eligible) else updated).sort_values(['amex_metric','roc_auc'],ascending=False).iloc[0].trial_id
    if trial_id==best_id:
        model.booster_.save_model(BEST_MODEL_PATH,num_iteration=model.best_iteration_)
        pd.DataFrame({'customer_ID':valid_ids,'target':y_valid.to_numpy(),'probability':probability.astype('float32'),'trial_id':trial_id}).to_parquet(BEST_PREDICTIONS_PATH,index=False,compression='zstd')
        print('NEW BEST MODEL:',trial_id)
    print('DONE',row); del model,probability; gc.collect(); return row


## 3. T0 — Extend the tree budget

The reference configuration is rerun with a larger `n_estimators` ceiling. Early stopping determines whether the previous cap constrained training. Matching the prior score confirms that the extended budget alone does not change the reference result.


In [ ]:
run_trial('T0_extended_reference','reference',{})


## 4. Stage 1 — Tree complexity

The search varies `num_leaves` and `min_child_samples`. More leaves allow richer interactions, while larger leaves regularize the tree by requiring more observations per terminal region.

Candidates are filtered with an AUC tolerance before moving forward so an isolated improvement in the top-weighted AMEX component does not conceal a material loss in global ranking.


In [ ]:
STRUCTURE_TRIALS=[('T1_leaves15_child100',{'num_leaves':15,'min_child_samples':100}),('T2_leaves31_child50',{'num_leaves':31,'min_child_samples':50}),('T3_leaves31_child200',{'num_leaves':31,'min_child_samples':200}),('T4_leaves63_child50',{'num_leaves':63,'min_child_samples':50}),('T5_leaves63_child100',{'num_leaves':63,'min_child_samples':100}),('T6_leaves63_child200',{'num_leaves':63,'min_child_samples':200})]
for trial_id,changes in STRUCTURE_TRIALS: run_trial(trial_id,'structure',changes)
stage1=load_results(); candidates=stage1[stage1.stage.isin(['reference','structure']) & (stage1.roc_auc>=REFERENCE['roc_auc']-.0001)]
if not len(candidates): candidates=stage1[stage1.stage.isin(['reference','structure'])]
best_structure=candidates.sort_values(['amex_metric','roc_auc'],ascending=False).iloc[0]
STRUCTURE={'num_leaves':int(best_structure.num_leaves),'min_child_samples':int(best_structure.min_child_samples)}
print('Selected structure:',STRUCTURE,'from',best_structure.trial_id)


## 5. Stage 2 — Sampling and regularization

Starting from the strongest structure, trials test row subsampling, column subsampling, L1 regularization, and stronger L2 regularization. These controls can reduce variance when the feature space is wide and correlated.


In [ ]:
SAMPLING_TRIALS=[('T7_bagging08',STRUCTURE|{'subsample':.8,'subsample_freq':1}),('T8_colsample07',STRUCTURE|{'colsample_bytree':.7}),('T9_colsample09',STRUCTURE|{'colsample_bytree':.9}),('T10_lambda5',STRUCTURE|{'reg_lambda':5.}),('T11_alpha01_lambda5',STRUCTURE|{'reg_alpha':.1,'reg_lambda':5.})]
for trial_id,changes in SAMPLING_TRIALS: run_trial(trial_id,'sampling_regularization',changes)


## 6. Stage 3 — Histogram resolution

The best eligible configuration is retested with `max_bin=255`. More bins can represent finer thresholds but increase memory, training time, and the opportunity to fit noise. The comparison asks whether that extra resolution produces a meaningful validation gain.


In [ ]:
before_bin=load_results(); eligible=before_bin[before_bin.roc_auc>=REFERENCE['roc_auc']-.0001]
best=(eligible if len(eligible) else before_bin).sort_values(['amex_metric','roc_auc'],ascending=False).iloc[0]
best_changes={k:(int(best[k]) if k in ['num_leaves','min_child_samples','subsample_freq','max_bin','n_estimators'] else float(best[k])) for k in PARAM_COLUMNS if k not in ['learning_rate']}
best_changes['max_bin']=255
run_trial('T12_maxbin255','binning',best_changes)


## 7. Rank trials and nominate CV candidates

Trials are ranked primarily by the official AMEX metric, with an AUC guardrail. Holdout ranking is only a screening device: repeatedly selecting on one validation set can overfit the experiment process itself.

T11 and T8 are therefore advanced to the same three-fold protocol as the reference model before a final decision is made.


In [ ]:
results=load_results().sort_values(['amex_metric','roc_auc'],ascending=False).reset_index(drop=True); results['rank']=np.arange(1,len(results)+1)
display(results[['rank','trial_id','stage','roc_auc','amex_metric','delta_auc_vs_04_5','delta_amex_vs_04_5','best_iteration','training_seconds',*PARAM_COLUMNS]])
cv_candidates=results[(results.delta_amex_vs_04_5>=.0003)|(results.delta_auc_vs_04_5>=.0001)].head(3)
summary={'reference_04_5':REFERENCE,'best_trial':results.iloc[0].to_dict(),'recommended_cv_candidates':cv_candidates.to_dict(orient='records'),'trial_count':len(results),'total_training_seconds':float(results.training_seconds.sum()),'selection_rule':'rank by AMEX metric, prefer AUC >= reference - 0.0001'}
SUMMARY_PATH.write_text(json.dumps(summary,indent=2,default=str),encoding='utf-8')
print('Recommended CV candidates:'); display(cv_candidates[['trial_id','roc_auc','amex_metric','delta_auc_vs_04_5','delta_amex_vs_04_5']])
for path in [RESULTS_PATH,BEST_MODEL_PATH,BEST_PREDICTIONS_PATH,SUMMARY_PATH]: print(path.name,'->',path.resolve())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Holdout trade-off across completed trials
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=results, x='roc_auc', y='amex_metric', hue='stage', size='training_seconds',
                sizes=(60, 220), alpha=0.85, ax=ax)
for _, row in results.iterrows():
    ax.annotate(row['trial_id'].split('_')[0], (row['roc_auc'], row['amex_metric']),
                xytext=(4, 3), textcoords='offset points', fontsize=8)
ax.axvline(REFERENCE['roc_auc'], color='0.55', linestyle='--', linewidth=1)
ax.axhline(REFERENCE['amex_metric'], color='0.55', linestyle='--', linewidth=1)
ax.set(title='Hyperparameter trials: ranking quality, AMEX metric, and runtime',
       xlabel='Validation ROC-AUC', ylabel='Validation AMEX metric')
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
sns.despine()
plt.tight_layout()
plt.show()
